<a href="https://colab.research.google.com/github/karandhariwal/FineTune_Llama-2-7b/blob/main/FineTune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
 !pip install -q accelerate peft bitsandbytes transformers trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.8 MB/s eta 0:00:00


In [ ]:
!pip install -q trl
import os
import torch
from datasets import load_dataset
from transformers import(
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

In [ ]:
# the model that you want to train from hugging face hub
model_name= "NousResearch/Llama-2-7b-chat-hf"

#the instruction dataset to use
dataset_name= "mlabonne/guanaco-llama2-1k"

#fine tuned model name
new_model="Llama-2-7b-chat-finetune"


In [ ]:
##Qlora Parametes

#lora attention dimention
lora_r = 64

#alfa parameter for lora scaling
lora_alpha = 16

#dropout propability for lora layers
lora_dropout = 0.1

##bitsandbytes parameters

# activate 4 bit precision base model loading
use_4bit = True

#compute dtype for 4-bit base models
bnb_4bit_compute_dtype = "float16"

#quantization type (fp4 or nf4)
bnb_4bit_quant_type = "nf4"

#Activate nested quantization for 4-bit base models (double quantization)
use_nested_quant = False


In [ ]:
# Training Arguments Parameters
output_dir = "./results"

#number of training epochs
num_train_epochs = 1

# Enable fp16/bf16 traing (set bf16 to true with an A100)
fp16 = False
bf16 = False

#Batch size per GPU for training
per_device_eval_batch_size = 4

#Batch size per GPU for evaluation
per_device_eval_batch_size = 4

#number of update steps to accumulate the gradients for
gradient_checkpointing = True

#Maximum gradient normal (gradient clipping)
max_grad_norm = 0.3

#initial learing rate (AdamW optimizer)
learning_rate = 2e-4

#Weight decay to apply to all layers except bias/LayerNorm weights
weight_decay = 0.001

#optimizer to use
optim = "paged_adamw_32bit"

#learing rate schedule
lr_scheduler_type = "cosine"

#number of training steps (override num_train_epochs)
max_steps = -1

#ratio of steps for a linear warmup (from 0 to learning rate)
warmup_ratio = 0.03

#group sequences into batches with same length
#Save money and speeds up training considerably
group_by_length = True

#Save checkpoint every X update steps
save_steps = 0

#log every X update steps
logging_steps = 25

In [ ]:
#SFT Parameters

#maximum sequence length to use
max_seq_length = None

#Pack multiple short examples in the same input sequence to increase efficiency
packing = False

#Load the entire model on the entire GPU 0
device_map = {"":0}



In [ ]:
from datasets import load_dataset

# Load the dataset (you can process it here)
dataset = load_dataset(dataset_name, split="train")

#load tokenizer and model with QLoRA configuration
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit = use_4bit,
    bnb_4bit_quant_type = bnb_4bit_quant_type,
    bnb_4bit_compute_dtype = compute_dtype,
    bnb_4bit_use_double_quant = use_nested_quant
)

README.md:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

data/train-00000-of-00001-9ad84bb9cf65a4(…): reconstructing file:   0%|          |  0.00B /  967kB            

data/train-00000-of-00001-9ad84bb9cf65a4(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
#check gpu compatibilty with bfloat16
if compute_dtype == torch.float16 and use_4bit:
  major, _ = torch.cuda.get_device_capability()

  if major  >= 8:
    print("=" * 80)
    print("Your GPU supports bfloat16: accelerate training with bf16=True")
    print("=" * 80)

#Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map = device_map
)

model.config.use_cache = False
model.config.pretraining_tp = 1


#load Llama tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code = True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" #Fix weird overflow issue with fp16 training

#Load Lora configuration
peft_config = LoraConfig(
    lora_alpha = lora_alpha,
    lora_dropout = lora_dropout,
    r= lora_r,
    bias = "none",
    task_type = "CAUSAL_LM"
)

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

In [ ]:
#set training parameters
training_arguments = TrainingArguments(
    output_dir = output_dir,
    num_train_epochs = num_train_epochs,
    per_device_train_batch_size = per_device_train_batch_size,
    gradient_accumulation_steps = gradient_accumulation_steps,
    optim = optim,
    save_steps = save_steps,
    logging_steps = logging_steps,
    learning_rate = learning_rate,
    weight_decay = weight_decay,
    fp16 = fp16,
    bf16 = bf16,
    max_grad_norm = max_grad_norm,
    max_steps = max_steps,
    warmup_ratio = warmup_ratio,
    # Removed 'group_by_length' as it's not a valid argument for TrainingArguments
    lr_scheduler_type = lr_scheduler_type,
    report_to = "tensorboard"
)


#set supervised fine tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset = dataset,
    peft_config = peft_config,
    # Removed 'dataset_text_field' as it's no longer a valid argument for SFTTrainer
    # Removed 'max_seq_length' as it's no longer a valid argument for SFTTrainer
    tokenizer = tokenizer,
    args = training_arguments,
    packing = packing
)

#Train model

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'tokenizer'